<a href="https://colab.research.google.com/github/mmariannaholanda/ia_identificacao_de_digitos/blob/main/RECONHECIMENTO_DE_DIGITOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# imports
import pandas as pd
import numpy as np

# datasets
from torchvision import datasets, transforms
# transforms = permite transformar imagens

dados_treino = datasets.MNIST(root="dados", train=True, download=True, transform=transforms.ToTensor())
dados_teste = datasets.MNIST(root="dados", train=False, download=True, transform=transforms.ToTensor())
# root="dados": pasta onde os arquivos do conjunto de dados serão armazenados
# train=True: indica que queremos os dados de treinamento
# transform=transforms.ToTensor(): converte as imagens para tensores do PyTorch


# sobre o bancode: images de shape: (28, 28, 1), dtype=uint8: tipo numérico utilizado para representar os pixels
print(len(dados_treino), len(dados_teste))

imagem, rotulo = dados_treino[0]
# imagem = recebe os pixels da imagem; rotulo = recebe o número correto representado nela

print(imagem.shape, rotulo)

100%|██████████| 9.91M/9.91M [00:01<00:00, 6.40MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 347kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.14MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.3MB/s]


60000 10000
torch.Size([1, 28, 28]) 5


o que sao tensores?? computador não enxerga a imagem como nós enxergamos. Ele representa cada pixel através de um número de 0 a 256
- o PyTorch trabalha com tensores para realizar suas operações matemáticas - tensor é uma estrutura usada para armazenar e organizar números em uma ou mais dimensões


São 60000 imagens de treino e 10000 de teste, MNIST não é uma tabela
MNIST é uma lista de pares de imagem, cada uma com uma posição, de 0 a 59999
1 posição devolve 2 coisas: tensor (imagem) e um rótulo (número)

agora deveria ser a etapa de padronizar, que é diferente no contexto de imagem...
cada pixel vai de 0 a 255, nossas imagens tem 784 pixels
ToTensor() pega e dividde todo mundo por 255, ou seja, agr a escala fica de 0 a 1
depois disso a gente padroniza - fNo MNIST, é a mesma conta
Cada imagem tem 784 pixels.
São 60000 imagens de treino.
Total de pixels: 784 × 60000 = 47.040.000.
Soma todos os valores, divide por 47.040.000, e dá 0,1307.

In [ ]:
# pre processamento de dados em ML

# media e desvio
media = dados_treino.data.float().mean() / 255
desvio = dados_treino.data.float().std() / 255

# standardization
treino_padronizado = (dados_treino.data.float() - media) / desvio
teste_padronizado = (dados_teste.data.float() - media) / desvio

In [ ]:
# baseline
contagem_treino = torch.bincount(dados_treino.targets)
print(contagem_treino)
classe_majoritaria = contagem_treino.argmax()

contagem_teste = torch.bincount(dados_teste.targets)

baseline = contagem_teste[classe_majoritaria] / len(dados_teste)
print(baseline)

NameError: name 'torch' is not defined

o
que faz	e por que :

**Conv2d**	desliza kernels pela imagem	enxerga vizinhança e reaproveita pesos,

**ReLU**	zera os negativos	a mesma razão: não linearidade,

**MaxPool2d**	guarda o maior valor de cada região	encolhe a imagem e tolera pequenos deslocamentos,

**Flatten**	achata o resultado em lista	para entrar na parte final,

**Linear**	cria uma camada linear de uma rede neural ( entradas, saidas)


agora vem 2 etapas novas...
1- MLP: 1) desenhar a rede, 2) medir o erro, 3) separar em lotes, 4) treinar, 5) avaliar

2- CNN: depois, trocando o começo da rede por kernels

In [ ]:
# montando a base de ref da rede

# faço um reshape pq antes treino_padronizado tinha formato (60000, 28, 28), ou seja, cada imagem era uma matriz 28,28
# com reshape cada imagem virou uma fila de 784 numeros e fica melhor de trabalhar

# atencao: o pixel que estava logo acima de outro passa a ficar 28 posições longe dele na fila. A rede não sabe mais quem era vizinho de quem
# É essa a limitação do MLP que a CNN resolve com os kernels, que deslizam pela grade sem achatá-la
entradas_treino = treino_padronizado.reshape(60000,784)
respostas_treino = dados_treino.targets

entradas_teste = teste_padronizado.reshape(10000, 784)
respostas_teste = dados_teste.targets

import torch.nn as nn
import torch

camada_escondida = nn.Linear(784, 128) # 784 entradas e 128 saídas
# essa parte é interessante pq o PyTorch sorteia 784 × 128 pesos aleatórios e 128 vieses
# e guarda dentro do objeto camada_escondida sem prcisar de np.random.randn

relu = nn.ReLU()

saida_escondida = camada_escondida(entradas_treino)
saida_relu = relu(saida_escondida)

pesos = camada_escondida.weight
vieses = camada_escondida.bias

conta = entradas_treino @ pesos.T + vieses

# saida_relu, com formato (60000, 128). São 60000 imagens, e cada uma virou 128 padrões
# quero: 10 números por imagem, uma nota para cada dígito (0 a 9). A maior nota será o palpite da rede
camada_saida = nn.Linear(128, 10)  # 128 entradas (os padrões) e 10 saídas (as notas)

saida_notas = camada_saida(saida_relu)


# nn.Sequential faz o papel da sua função propagar
modelo = nn.Sequential(camada_escondida, relu, camada_saida)

saida_notas_modelo = modelo(entradas_treino)

total_parametros = sum(p.numel() for p in modelo.parameters())
print(total_parametros)

funcao_perda = nn.CrossEntropyLoss()
perda = funcao_perda(saida_notas, respostas_treino)


In [ ]:
import matplotlib.pyplot as plt

pesos_neuronio = camada_escondida.weight[0].detach().reshape(28, 28)
plt.imshow(pesos_neuronio, cmap="gray")
plt.show()

In [ ]:
torch.manual_seed(42)  # fixa o sorteio dos pesos


# calculamos o erro da rede neural
perda_antes = funcao_perda(modelo(entradas_treino), respostas_treino)
perda_depois = funcao_perda(modelo(entradas_treino), respostas_treino)
print(perda_antes.item(), perda_depois.item())

# calculamos as derivadas parciais da função de perda em relação a cada peso e viés da rede
perda_antes.backward()  # executa a retropropagação (backpropagation)


# utiliza os gradientes encontrados na retropropagação para atualizar os pesos e vieses, tentando diminuir o erro
# taxa de aprendizado = 0.1
# optim: módulo que contém os algoritmos de otimização
# SGD: algoritmo Stochastic Gradient Descent (Descida do Gradiente Estocástico)
# modelo.parameters(): obtém os parâmetros treináveis do modelo, ou seja, pesos e vieses
# otimizador = ATUALIZACAO DOS PARAMETROS
otimizador = torch.optim.SGD(modelo.parameters(), lr=0.1)
otimizador.zero_grad() # apaga os gradientes antigos para n ficar somando
otimizador.step()  # atualização dos parâmetros utilizando os gradientes calculados anteriormente
# zero_grad()  →  backward()  →  step()


In [ ]:
# loop
taxa_aprendizado = 0.1
quantidade_passos = 5000

otimizador = torch.optim.SGD(modelo.parameters(), lr=taxa_aprendizado)
funcao_perda = nn.CrossEntropyLoss()

historico_perda = []  # lista vazia para guardar a perda de cada passo

for passo in range(quantidade_passos):

    # 1) propagar e medir o erro com os pesos atuais
    notas = modelo(entradas_treino)
    perda = funcao_perda(notas, respostas_treino)
    historico_perda.append(perda.item())

    # 2) ajustar: limpar, calcular gradientes, andar
    otimizador.zero_grad()
    perda.backward()
    otimizador.step()

    if passo % 10 == 0:
        print(f"passo {passo}: perda = {perda.item():.4f}")

print(f"perda final: {historico_perda[-1]:.4f}")

In [ ]:
# implementando np.savez
# media e desvio eram do pandas


pesos_escondida = modelo[0].weight.detach().numpy()
vies_escondida = modelo[0].bias.detach().numpy()
pesos_saida = modelo[2].weight.detach().numpy()
vies_saida = modelo[2].bias.detach().numpy()
print("\n========================================================================\n") # organizando



np.savez(
    "reconhecimento_digitos.npz",  # função do NumPy que salva vários arrays em um único arquivo .npz
    pesos_escondida=pesos_escondida,
    vies_escondida=vies_escondida,
    pesos_saida=pesos_saida,
    vies_saida=vies_saida,
    media=media.item(),  # extrair o valor numérico de um tensor
    desvio=desvio.item(),
)
print("modelo salvo ok\n")
print("conferindo...")

modelo = np.load( "reconhecimento_digitos.npz")
for nome in modelo.files:
    print(nome, modelo[nome].shape)

    # vai me dizer linhas e colunas
    # media, desvio e colunas serão apenas vetores e o resto matrizes


# dando uma olhada em como ficou cada formato visualmente
print("\n=========================================================================\n")
np.set_printoptions(precision=3, suppress=True, linewidth=150)

for nome in arquivo_modelo.files:
    print("=====", nome, arquivo_modelo[nome].shape)
    print(arquivo_modelo[nome])

In [ ]:
# gráfico da perda
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(historico_perda)
plt.xlabel("Passo")
plt.ylabel("Perda (entropia cruzada)")
plt.title("Perda do treino a cada passo")
plt.grid(True)

plt.savefig("perda_treino.png", dpi=150, bbox_inches="tight")


plt.show()

In [ ]:
# acuracia

# torch.no_grad() é utilizado para desativar o cálculo e o armazenamento dos gradientes durante determinado trecho do código
with torch.no_grad():  # corrigido: so estamos medindo, sem calcular gradientes
    notas_treino = modelo(entradas_treino)
    previsoes_treino = notas_treino.argmax(dim=1)
    acuracia_treino = (previsoes_treino == respostas_treino).float().mean().item()

    # no teste (dados que a rede nunca viu)
    notas_teste = modelo(entradas_teste)
    previsoes_teste = notas_teste.argmax(dim=1)
    acuracia_teste = (previsoes_teste == respostas_teste).float().mean().item()


acuracia_treino_percentual = acuracia_treino * 100
acuracia_teste_percentual = acuracia_teste * 100
baseline_percentual = baseline.item() * 100

print(f"Acurácia no treino: {acuracia_treino_percentual:.3f} %")
print(f"Acurácia no teste:  {acuracia_teste_percentual:.3f} %")
print(f"Baseline (sempre o digito mais comum): {baseline_percentual:.3f} %")
print(f"Ganho sobre o baseline: {acuracia_teste_percentual - baseline_percentual:.3f} pontos")

In [ ]:
# aqui eu faço o recall:

recalls = []

for digito in range(10):
    verdadeiros_positivos = ((previsoes_teste == digito) & (respostas_teste == digito)).sum().item()
    falsos_negativos = ((previsoes_teste != digito) & (respostas_teste == digito)).sum().item()
    falsos_positivos = ((previsoes_teste == digito) & (respostas_teste != digito)).sum().item()
    verdadeiros_negativos = ((previsoes_teste != digito) & (respostas_teste != digito)).sum().item()

    recall_digito = verdadeiros_positivos / (verdadeiros_positivos + falsos_negativos)
    recall_digito_percentual = recall_digito * 100
    recalls.append(recall_digito)

    print(f"===== Dígito {digito} =====")
    print(f"Verdadeiros positivos: {verdadeiros_positivos}")
    print(f"Falsos negativos: {falsos_negativos}")
    print(f"Falsos positivos: {falsos_positivos}")
    print(f"Verdadeiros negativos: {verdadeiros_negativos}")
    print(f"Soma das quatro: {verdadeiros_positivos + falsos_negativos + falsos_positivos + verdadeiros_negativos}")
    print(f"Recall: {recall_digito_percentual:.3f} %\n")

recall_medio = sum(recalls) / len(recalls)
print(f"Recall médio dos 10 dígitos: {recall_medio * 100:.3f} %")

In [ ]:
matriz_confusao = torch.zeros(10, 10, dtype=torch.int64)
# 10 linhas (real) x 10 colunas (previsto), tudo zero e so numeros inteiros

for digito_real, digito_previsto in zip(respostas_teste, previsoes_teste):
    matriz_confusao[digito_real, digito_previsto] += 1
# para cada imagem do teste, soma 1 na celula [real, previsto]

print(matriz_confusao)
print("Soma da matriz:", matriz_confusao.sum().item())

# agora eu desenho
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 7))
plt.imshow(matriz_confusao, cmap="Blues")

nomes_classes = [str(d) for d in range(10)]
plt.xticks(range(10), nomes_classes)
plt.yticks(range(10), nomes_classes)
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.title("Matriz de confusão - Dígitos")

for linha in range(10):
    for coluna in range(10):
        valor = matriz_confusao[linha, coluna].item()
        cor_texto = "white" if valor > matriz_confusao.max().item() / 2 else "black"
        plt.text(coluna, linha, str(valor), ha="center", va="center", color=cor_texto, fontsize=9)

plt.savefig("matriz_confusao_digitos.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt

arquivo_enviado = files.upload()  # abre o botao para escolher a imagem
nome_arquivo = list(arquivo_enviado.keys())[0]

imagem_original = Image.open(nome_arquivo).convert("L")  # "L" = tons de cinza
imagem_reduzida = imagem_original.resize((28, 28))
pixels = torch.tensor(np.array(imagem_reduzida), dtype=torch.float32)  # 28x28, de 0 a 255

if pixels.mean() > 127:  # fundo claro: inverte para ficar como o MNIST
    pixels = 255 - pixels

entrada_nova = ((pixels - media) / desvio).reshape(1, 784)  # mesma padronizacao do treino

with torch.no_grad():
    notas = modelo(entrada_nova)
    probabilidades = torch.softmax(notas, dim=1)

palpite = notas.argmax(dim=1).item()
confianca = probabilidades[0, palpite].item() * 100

print(f"Palpite da rede: {palpite} (confiança {confianca:.1f} %)")

plt.imshow(pixels, cmap="gray")
plt.title(f"O que a rede viu. Palpite: {palpite}")
plt.show()